In [16]:
# ============================================================
# SMART EXPENSE MANAGEMENT
# REAL-WORLD TEXT CLASSIFICATION
# ============================================================

# -----------------------------
# 1. IMPORT LIBRARIES
# -----------------------------

import pandas as pd
import numpy as np
import re
import pickle

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion
from sklearn.metrics import accuracy_score, classification_report


# ============================================================
# 2. LOAD DATASET
# ============================================================

df = pd.read_csv("smart_expense_upgraded_7000.csv")

# Keep only required columns
df = df[["description", "category"]].copy()

# Remove missing values
df = df.dropna()

# Convert to string
df["description"] = df["description"].astype(str)
df["category"] = df["category"].astype(str)

# Clean spaces
df["description"] = df["description"].str.lower().str.strip()

# Remove duplicate descriptions
df = df.drop_duplicates(subset="description")

print("Dataset shape:", df.shape)
print("\nCategory distribution:")
print(df["category"].value_counts())


# ============================================================
# 3. ADD GOOD REAL-WORLD EXAMPLES
# ============================================================

extra_data = [

    # ---------------- SHOPPING ----------------
    ("bought new shoes", "Shopping"),
    ("purchased jeans online", "Shopping"),
    ("bought a new shirt", "Shopping"),
    ("purchased a backpack", "Shopping"),
    ("bought headphones", "Shopping"),
    ("bought a laptop", "Shopping"),
    ("purchased a mobile phone", "Shopping"),
    ("bought a phone cover", "Shopping"),
    ("purchased a smartwatch", "Shopping"),
    ("bought a wrist watch", "Shopping"),
    ("bought a wallet", "Shopping"),
    ("purchased sunglasses", "Shopping"),
    ("bought a jacket", "Shopping"),
    ("purchased a handbag", "Shopping"),
    ("bought furniture for home", "Shopping"),
    ("purchased a chair", "Shopping"),
    ("bought a table", "Shopping"),
    ("bought kitchen items", "Shopping"),
    ("purchased home decoration", "Shopping"),
    ("bought a power bank", "Shopping"),
    ("purchased a charger", "Shopping"),
    ("bought a computer mouse", "Shopping"),
    ("purchased a keyboard", "Shopping"),
    ("bought a printer", "Shopping"),
    ("bought clothes from mall", "Shopping"),
    ("purchased clothes from an online store", "Shopping"),
    ("bought cosmetics", "Shopping"),
    ("purchased perfume", "Shopping"),
    ("bought a gift", "Shopping"),
    ("bought a bicycle", "Shopping"),

    # ---------------- BILLS ----------------
    ("paid electricity bill", "Bills"),
    ("electricity bill payment", "Bills"),
    ("paid water bill", "Bills"),
    ("water bill payment", "Bills"),
    ("paid gas bill", "Bills"),
    ("gas bill payment", "Bills"),
    ("paid internet bill", "Bills"),
    ("internet bill payment", "Bills"),
    ("paid broadband bill", "Bills"),
    ("broadband payment", "Bills"),
    ("paid mobile bill", "Bills"),
    ("mobile recharge", "Bills"),
    ("paid phone bill", "Bills"),
    ("telephone bill payment", "Bills"),
    ("paid dth recharge", "Bills"),
    ("dth recharge", "Bills"),
    ("paid house rent", "Bills"),
    ("monthly rent payment", "Bills"),
    ("paid society maintenance", "Bills"),
    ("society maintenance payment", "Bills"),

    # ---------------- FOOD ----------------
    ("ordered pizza", "Food"),
    ("bought burger", "Food"),
    ("had dinner at restaurant", "Food"),
    ("had lunch at cafe", "Food"),
    ("ordered food online", "Food"),
    ("bought vegetables", "Food"),
    ("bought fruits", "Food"),
    ("bought groceries", "Food"),
    ("bought bread and milk", "Food"),
    ("ordered biryani", "Food"),
    ("bought snacks", "Food"),
    ("had coffee at cafe", "Food"),
    ("had tea", "Food"),
    ("bought ice cream", "Food"),
    ("ordered sandwich", "Food"),
    ("restaurant bill", "Food"),
    ("food delivery order", "Food"),
    ("breakfast at restaurant", "Food"),
    ("dinner with friends", "Food"),
    ("ordered noodles", "Food"),

    # ---------------- ENTERTAINMENT ----------------
    ("bought movie ticket", "Entertainment"),
    ("movie ticket booking", "Entertainment"),
    ("went to cinema", "Entertainment"),
    ("paid for movie", "Entertainment"),
    ("concert ticket", "Entertainment"),
    ("concert booking", "Entertainment"),
    ("paid for netflix", "Entertainment"),
    ("netflix subscription", "Entertainment"),
    ("spotify subscription", "Entertainment"),
    ("amazon prime subscription", "Entertainment"),
    ("youtube premium subscription", "Entertainment"),
    ("bought a video game", "Entertainment"),
    ("gaming purchase", "Entertainment"),
    ("went to amusement park", "Entertainment"),
    ("theme park ticket", "Entertainment"),
    ("entertainment event ticket", "Entertainment"),
    ("movie theatre ticket", "Entertainment"),
    ("paid for gaming subscription", "Entertainment"),
    ("web series subscription", "Entertainment"),
    ("music subscription", "Entertainment"),

    # ---------------- EDUCATION ----------------
    ("paid college fees", "Education"),
    ("paid semester fees", "Education"),
    ("college admission fee", "Education"),
    ("paid school fees", "Education"),
    ("university fees", "Education"),
    ("bought college textbook", "Education"),
    ("purchased study books", "Education"),
    ("bought educational book", "Education"),
    ("paid online course fee", "Education"),
    ("coding course payment", "Education"),
    ("paid exam fees", "Education"),
    ("tuition fees", "Education"),
    ("coaching fees", "Education"),
    ("bought study material", "Education"),
    ("college registration fee", "Education"),
    ("paid university admission fee", "Education"),
    ("bought academic books", "Education"),
    ("paid training course fee", "Education"),
    ("online learning course", "Education"),
    ("college project material", "Education"),

    # ---------------- HEALTHCARE ----------------
    ("visited doctor", "Healthcare"),
    ("doctor consultation fee", "Healthcare"),
    ("doctor appointment", "Healthcare"),
    ("hospital bill", "Healthcare"),
    ("paid hospital fees", "Healthcare"),
    ("bought medicines", "Healthcare"),
    ("medicine purchase", "Healthcare"),
    ("pharmacy bill", "Healthcare"),
    ("medical test payment", "Healthcare"),
    ("blood test", "Healthcare"),
    ("health checkup", "Healthcare"),
    ("medical treatment", "Healthcare"),
    ("dental treatment", "Healthcare"),
    ("dentist appointment", "Healthcare"),
    ("prescription medicine", "Healthcare"),
    ("medical scan", "Healthcare"),
    ("x ray test", "Healthcare"),
    ("medical consultation", "Healthcare"),
    ("pharmacy purchase", "Healthcare"),
    ("hospital consultation", "Healthcare"),

    # ---------------- TRANSPORTATION ----------------
    ("filled petrol", "Transportation"),
    ("petrol for my bike", "Transportation"),
    ("bought diesel", "Transportation"),
    ("fuel payment", "Transportation"),
    ("uber ride", "Transportation"),
    ("ola ride", "Transportation"),
    ("rapido ride", "Transportation"),
    ("auto rickshaw fare", "Transportation"),
    ("taxi fare", "Transportation"),
    ("cab payment", "Transportation"),
    ("bus ticket", "Transportation"),
    ("bus fare", "Transportation"),
    ("train ticket", "Transportation"),
    ("metro ticket", "Transportation"),
    ("flight ticket", "Transportation"),
    ("parking fee", "Transportation"),
    ("toll payment", "Transportation"),
    ("rickshaw fare", "Transportation"),
    ("bike fuel", "Transportation"),
    ("cab ride to office", "Transportation")
]


extra_df = pd.DataFrame(
    extra_data,
    columns=["description", "category"]
)

# Add examples
df = pd.concat([df, extra_df], ignore_index=True)

# Remove duplicates again
df = df.drop_duplicates(subset="description").reset_index(drop=True)

print("\nAfter adding real-world examples:")
print(df["category"].value_counts())


# ============================================================
# 4. TRAIN / TEST SPLIT
# ============================================================

X = df["description"]
y = df["category"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# ============================================================
# 5. WORD + CHARACTER TF-IDF
# ============================================================

word_tfidf = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True,
    max_features=15000
)

char_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=1,
    sublinear_tf=True,
    max_features=15000
)

features = FeatureUnion([
    ("word", word_tfidf),
    ("char", char_tfidf)
])


# ============================================================
# 6. CREATE FEATURES
# ============================================================

X_train_vec = features.fit_transform(X_train)
X_test_vec = features.transform(X_test)

print("\nTraining feature shape:", X_train_vec.shape)


# ============================================================
# 7. TRAIN MODEL
# ============================================================

model = LogisticRegression(
    max_iter=3000,
    C=3
)

model.fit(X_train_vec, y_train)


# ============================================================
# 8. MODEL TEST
# ============================================================

y_pred = model.predict(X_test_vec)

accuracy = accuracy_score(y_test, y_pred)

print("\n===================================")
print("MODEL RESULT")
print("===================================")

print("Accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


# ============================================================
# 9. SIMPLE REAL-WORLD SAFETY RULES
# ============================================================

def rule_based_category(text):

    text = text.lower()

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # ---------------- BILLS ----------------
    bill_words = [
        "electricity bill",
        "water bill",
        "gas bill",
        "internet bill",
        "broadband bill",
        "mobile bill",
        "phone bill",
        "telephone bill",
        "dth recharge",
        "mobile recharge",
        "jio recharge",
        "airtel recharge",
        "vi recharge",
        "house rent",
        "rent payment",
        "society maintenance"
    ]

    if any(word in text for word in bill_words):
        return "Bills"


    # ---------------- HEALTHCARE ----------------
    health_words = [
        "doctor",
        "hospital",
        "medicine",
        "medicines",
        "pharmacy",
        "medical",
        "prescription",
        "dentist",
        "dental",
        "health checkup",
        "blood test",
        "x ray",
        "medical test",
        "lab test"
    ]

    if any(word in text for word in health_words):
        return "Healthcare"


    # ---------------- TRANSPORTATION ----------------
    transport_words = [
        "petrol",
        "diesel",
        "cng",
        "fuel",
        "uber",
        "ola",
        "rapido",
        "rickshaw",
        "auto fare",
        "cab ride",
        "taxi",
        "bus ticket",
        "bus fare",
        "train ticket",
        "metro ticket",
        "flight ticket",
        "parking fee",
        "toll payment"
    ]

    if any(word in text for word in transport_words):
        return "Transportation"


    # ---------------- ENTERTAINMENT ----------------
    entertainment_words = [
        "movie ticket",
        "movie booking",
        "cinema",
        "concert",
        "netflix",
        "spotify",
        "amazon prime",
        "youtube premium",
        "gaming",
        "video game",
        "game purchase",
        "amusement park",
        "theme park"
    ]

    if any(word in text for word in entertainment_words):
        return "Entertainment"


    # ---------------- EDUCATION ----------------
    education_words = [
        "college fees",
        "college fee",
        "semester fees",
        "semester fee",
        "school fees",
        "school fee",
        "university fees",
        "university fee",
        "tuition fee",
        "tuition fees",
        "coaching fee",
        "coaching fees",
        "exam fee",
        "exam fees",
        "course fee",
        "course fees",
        "college textbook",
        "college book",
        "study material",
        "college project"
    ]

    if any(word in text for word in education_words):
        return "Education"


    # ---------------- FOOD ----------------
    food_words = [
        "pizza",
        "burger",
        "biryani",
        "sandwich",
        "restaurant",
        "cafe",
        "coffee",
        "breakfast",
        "lunch",
        "dinner",
        "food delivery",
        "zomato",
        "swiggy",
        "groceries",
        "vegetables",
        "fruits",
        "snacks",
        "ice cream"
    ]

    if any(word in text for word in food_words):
        return "Food"


    # ---------------- SHOPPING ----------------
    shopping_words = [
        "shoes",
        "jeans",
        "shirt",
        "clothes",
        "jacket",
        "backpack",
        "headphones",
        "laptop",
        "mobile phone",
        "phone cover",
        "smartwatch",
        "watch",
        "wallet",
        "sunglasses",
        "handbag",
        "furniture",
        "chair",
        "table",
        "cosmetics",
        "perfume",
        "charger",
        "power bank",
        "keyboard",
        "mouse",
        "printer",
        "bicycle"
    ]

    if any(word in text for word in shopping_words):
        return "Shopping"


    # No strong rule found
    return None


# ============================================================
# 10. FINAL PREDICTION FUNCTION
# ============================================================

def predict_expense(text):

    text = str(text).lower().strip()

    # First check strong real-world rules
    rule_result = rule_based_category(text)

    if rule_result is not None:

        return {
            "expense": text,
            "category": rule_result,
            "confidence": "High",
            "method": "Real-world rule"
        }


    # Otherwise use ML model
    text_vec = features.transform([text])

    prediction = model.predict(text_vec)[0]

    probabilities = model.predict_proba(text_vec)[0]

    confidence = max(probabilities) * 100

    # If model is not confident
    if confidence < 45:

        return {
            "expense": text,
            "category": "Needs Review",
            "confidence": round(confidence, 2),
            "method": "ML - Low confidence"
        }

    return {
        "expense": text,
        "category": prediction,
        "confidence": round(confidence, 2),
        "method": "ML model"
    }


# ============================================================
# 11. TEST REAL-WORLD EXAMPLES
# ============================================================

test_expenses = [

    "I bought a new pair of shoes",
    "I purchased a laptop",
    "I bought a new mobile phone",

    "paid my electricity bill",
    "paid broadband bill",

    "ordered pizza for dinner",
    "had lunch at restaurant",

    "bought a movie ticket",
    "paid for Netflix",

    "paid my college semester fees",
    "bought a college textbook",

    "bought medicines from pharmacy",
    "visited the doctor",

    "filled petrol in my bike",
    "booked an Uber ride",

    "bought a chair for my room",
    "purchased headphones",
    "bought a watch",

    "paid water bill",
    "bought vegetables",
    "paid coaching fees",
    "bought a video game"
]


print("\n===================================")
print("REAL-WORLD TEST")
print("===================================")

for expense in test_expenses:

    result = predict_expense(expense)

    print("\nExpense :", expense)
    print("Category:", result["category"])
    print("Confidence:", result["confidence"])
    print("Method:", result["method"])


# ============================================================
# 12. CUSTOM USER INPUT
# ============================================================

print("\n===================================")
print("CUSTOM EXPENSE")
print("===================================")

while True:

    user_input = input(
        "\nEnter expense (type 'exit' to stop): "
    )

    if user_input.lower() == "exit":
        break

    result = predict_expense(user_input)

    print("\nPredicted Category:", result["category"])
    print("Confidence:", result["confidence"])
    print("Method:", result["method"])


# ============================================================
# 13. SAVE MODEL
# ============================================================

with open("expense_model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("expense_tf-idf.pkl", "wb") as f:
    pickle.dump(features, f)

print("\n===================================")
print("MODEL SAVED SUCCESSFULLY")
print("===================================")
print("expense_model.pkl")
print("expense_tf-idf.pkl")

Dataset shape: (7000, 2)

Category distribution:
category
Shopping          1000
Bills             1000
Food              1000
Entertainment     1000
Education         1000
Healthcare        1000
Transportation    1000
Name: count, dtype: int64

After adding real-world examples:
category
Shopping          1030
Bills             1020
Food              1020
Entertainment     1020
Education         1020
Healthcare        1020
Transportation    1020
Name: count, dtype: int64

Training feature shape: (5720, 9794)

MODEL RESULT
Accuracy: 100.0 %

Classification Report:
                precision    recall  f1-score   support

         Bills       1.00      1.00      1.00       204
     Education       1.00      1.00      1.00       204
 Entertainment       1.00      1.00      1.00       204
          Food       1.00      1.00      1.00       204
    Healthcare       1.00      1.00      1.00       204
      Shopping       1.00      1.00      1.00       206
Transportation       1.00      1.00  